# 🧪 Checkpoint 1 — Data & Geographic Assessment & Spatial Integration
## Project: Mérida Urban Intelligence | Team: Sunchos Team

### Objectives:
1. Inventory and profile raw data sources (Demographics, DENUE, AGEB Cartography, Crime).
2. Document geographic-unit decision (AGEB) and justification.
3. Demonstrate Point-to-Polygon spatial integration (Spatial Join).
4. List source variables required by analytical KPIs.

In [ ]:
import sys
from pathlib import Path

# Locate the repository whether Jupyter starts in root or notebooks/.
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'src' / 'etl' / 'extract.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from src.etl.config import CRS_WGS84, CRS_METRIC_MEXICO, DATA_RAW_DIR
from src.etl.extract import get_real_paths
from src.qa.audit_local_data import audit_geojson, DEFAULT_INPUT

print("✅ Libraries imported successfully.")

## 1. Data Source Inventory & Profiling
Scanning `data/raw/` for datasets:

In [ ]:
inventory = get_real_paths()
for category, path in inventory.items():
    if path is None:
        print(f'{category.upper()}: source absent; raw profiling pending.')
    else:
        print(f'{category.upper()}: {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024:.1f} KB)')

# Profile the committed file, independently of database access.
local_audit = audit_geojson(DEFAULT_INPUT)
print('Local features:', local_audit['features'])
display(pd.DataFrame(local_audit['properties']).T)
display(pd.DataFrame(local_audit['findings']))
print('Supabase not queried; topology and raw sources not validated.')

## 2. Geographic Unit Decision: AGEB Urbana de Mérida

### Justification:
- **Balance of Resolution**: Aggregation to AGEB reduces some limitations of block-level data; suppressed values still require auditing.
- **Source Compatibility**: Matches official INEGI Censo 2020 aggregations and DENUE coordinates.
- **Spatial Metric Accuracy**: Reprojected to EPSG:6372 / EPSG:32616 for precise area calculations ($km^2$).

## 3. Point-to-Polygon Spatial Integration Test
Testing spatial join between synthetic point coordinates and synthetic polygons. This is a method demonstration, not evidence of actual DENUE/crime coverage. With `within`, boundary points are excluded.

In [ ]:
# Create sample polygons (Mérida AGEBs mock/test)
poly1 = Polygon([(-89.65, 20.95), (-89.60, 20.95), (-89.60, 21.00), (-89.65, 21.00)])
poly2 = Polygon([(-89.60, 20.95), (-89.55, 20.95), (-89.55, 21.00), (-89.60, 21.00)])

gdf_sample_polys = gpd.GeoDataFrame(
    {
        'cvegeo': ['310500001001A', '310500001002A'],
        'nom_asentamiento': ['Centro', 'Itzimná'],
        'geometry': [poly1, poly2]
    },
    crs=CRS_WGS84
)

# Sample points (Incidents & Businesses)
df_sample_points = pd.DataFrame({
    'id': [101, 102, 103],
    'tipo': ['Comercio', 'Delito', 'Servicio'],
    'latitud': [20.97, 20.98, 20.96],
    'longitud': [-89.63, -89.58, -89.62]
})

# Execute Spatial Join
gdf_sample_points = gpd.GeoDataFrame(
    df_sample_points,
    geometry=gpd.points_from_xy(df_sample_points['longitud'], df_sample_points['latitud']),
    crs=CRS_WGS84
)
joined_res = gpd.sjoin(gdf_sample_points, gdf_sample_polys, how='inner', predicate='within')
assert len(joined_res) == 3
assert joined_res.set_index('id')['cvegeo'].to_dict() == {
    101: '310500001001A', 102: '310500001002A', 103: '310500001001A'
}
print("✅ Spatial Join Test Output:")
print(joined_res[['id', 'tipo', 'cvegeo', 'latitud', 'longitud']])